# 🤖 Teaching a Computer to Read Heartbeats

**Duration:** 90 minutes  
**Level:** Intermediate

---

## Welcome to Your First AI Project!

Have you ever wondered how your smartwatch knows when you're exercising? Or how your phone can recognize your face? They use **artificial intelligence (AI)** and **machine learning (ML)**!

Today, you're going to build your very own AI that can look at biosignals and tell whether someone is relaxed or stressed. No magic required - just some clever math and a bit of code!

### What You'll Learn:
- 🧠 What machine learning really means (in simple terms!)
- 📊 How to prepare data for AI
- 🎓 How to train your first AI model
- 🎯 How to test and improve your AI
- 📈 How to understand what your AI got right and wrong

Let's create some AI magic!

## 🤔 What is Machine Learning?

**Traditional Programming:**
- You tell the computer EXACTLY what to do
- Example: "If heart rate > 75, then person is active"

**Machine Learning:**
- You show the computer lots of examples
- The computer figures out the rules by itself!
- Example: Show it 100 heartbeats labeled "relaxed" or "active", and it learns the pattern

Think of it like learning to ride a bike:
- Traditional programming = Following exact instructions
- Machine learning = Practicing until you figure it out yourself

Let's get started!

## 📦 Step 1: Loading Our Tools

We'll use **scikit-learn (sklearn)** - a popular Python library for machine learning!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import plot_tree
import warnings
warnings.filterwarnings('ignore')

# Make our plots beautiful
plt.style.use('seaborn-v0_8-whitegrid')

print("✅ All tools loaded!")
print("   Ready to build your first AI!")

## 🧪 Step 2: Creating Our Dataset

First, we need data to teach our AI. We'll create synthetic biosignals for two states:
- **Relaxed** (label = 0)
- **Active/Stressed** (label = 1)

We'll generate LOTS of examples so our AI has plenty to learn from!

In [ ]:
def generate_heartbeat_signal(duration=30, sampling_rate=100, base_hr=70, variability=5, stress_level=0):
    """
    Generate a synthetic heartbeat signal.
    """
    t = np.linspace(0, duration, duration * sampling_rate)
    
    # Create heart rate that varies over time
    hr = base_hr + variability * np.sin(2 * np.pi * 0.1 * t)
    
    # Add random variations
    hrv_noise = np.random.normal(0, variability * (1 + stress_level), len(t))
    hr = hr + hrv_noise
    
    # Add stress component
    if stress_level > 0:
        stress_component = stress_level * 10 * np.sin(2 * np.pi * 0.5 * t)
        hr = hr + stress_component
    
    # Create the actual signal with peaks
    signal_data = np.zeros_like(t)
    idx = 0
    
    while idx < len(t):
        beats_per_second = hr[idx] / 60
        time_to_next_beat = 1 / beats_per_second
        
        peak_width = int(0.15 * sampling_rate)
        peak_start = idx
        peak_end = min(idx + peak_width, len(signal_data))
        
        peak_shape = signal.windows.gaussian(peak_width, std=peak_width/6)
        signal_data[peak_start:peak_end] = peak_shape[:peak_end-peak_start]
        
        idx += int(time_to_next_beat * sampling_rate)
    
    return signal_data

def extract_features(signal_data, sampling_rate=100):
    """
    Extract features from a biosignal.
    These features are what the AI will learn from!
    """
    features = {}
    
    # Basic statistics
    features['mean'] = np.mean(signal_data)
    features['std'] = np.std(signal_data)
    features['max'] = np.max(signal_data)
    features['min'] = np.min(signal_data)
    
    # Peak features
    peaks, _ = signal.find_peaks(signal_data, height=0.3, distance=50)
    features['num_peaks'] = len(peaks)
    features['avg_peak_height'] = np.mean(signal_data[peaks]) if len(peaks) > 0 else 0
    
    # Heart rate features
    if len(peaks) > 1:
        peak_intervals = np.diff(peaks) / sampling_rate
        heart_rates = 60 / peak_intervals
        features['avg_heart_rate'] = np.mean(heart_rates)
        features['heart_rate_std'] = np.std(heart_rates)
        features['min_hr'] = np.min(heart_rates)
        features['max_hr'] = np.max(heart_rates)
    else:
        features['avg_heart_rate'] = 0
        features['heart_rate_std'] = 0
        features['min_hr'] = 0
        features['max_hr'] = 0
    
    # Variability features
    differences = np.diff(signal_data)
    features['mean_diff'] = np.mean(np.abs(differences))
    features['std_diff'] = np.std(differences)
    
    return list(features.values())

print("🔧 Helper functions ready!")

In [ ]:
print("🧪 Generating training dataset...\n")
print("This might take a moment - we're creating 200 samples!\n")

# Generate lots of examples!
n_samples_per_class = 100
X = []  # Features (the input to our AI)
y = []  # Labels (the correct answers)

# Generate RELAXED samples (label = 0)
for i in range(n_samples_per_class):
    sig = generate_heartbeat_signal(
        duration=30,
        base_hr=65 + np.random.normal(0, 5),  # Vary the parameters a bit
        variability=3 + np.random.normal(0, 1),
        stress_level=0
    )
    features = extract_features(sig)
    X.append(features)
    y.append(0)  # Label: Relaxed

# Generate ACTIVE samples (label = 1)
for i in range(n_samples_per_class):
    sig = generate_heartbeat_signal(
        duration=30,
        base_hr=85 + np.random.normal(0, 5),
        variability=8 + np.random.normal(0, 1.5),
        stress_level=0.7 + np.random.normal(0, 0.2)
    )
    features = extract_features(sig)
    X.append(features)
    y.append(1)  # Label: Active/Stressed

# Convert to numpy arrays
X = np.array(X)
y = np.array(y)

print(f"✅ Dataset created!")
print(f"   Total samples: {len(X)}")
print(f"   Relaxed samples: {np.sum(y == 0)}")
print(f"   Active samples: {np.sum(y == 1)}")
print(f"   Features per sample: {X.shape[1]}")
print(f"\n📊 Each sample has {X.shape[1]} features that describe the heartbeat!")

## 🎯 Step 3: Preparing Data for AI (Train/Test Split)

Here's an important concept: We'll split our data into two parts:

**Training Set (80%):** The AI learns from these examples  
**Testing Set (20%):** We use these to see if the AI really learned (it hasn't seen these before!)

This is like studying for a test:
- Training data = practice problems you study
- Testing data = the actual test questions (new problems!)

If you only memorize the practice problems, you won't do well on new questions. Same with AI!

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,  # 20% for testing
    random_state=42,  # Makes results reproducible
    stratify=y  # Ensures equal proportions in both sets
)

print("📊 Data Split Complete!\n")
print(f"Training set: {len(X_train)} samples")
print(f"  - Relaxed: {np.sum(y_train == 0)}")
print(f"  - Active: {np.sum(y_train == 1)}")
print(f"\nTesting set: {len(X_test)} samples")
print(f"  - Relaxed: {np.sum(y_test == 0)}")
print(f"  - Active: {np.sum(y_test == 1)}")
print(f"\n💡 The AI will learn from the training set and be tested on the testing set!")

## 🎓 Step 4: Training Your First AI!

We'll use a **Decision Tree** - it's like a flowchart that asks questions:
- "Is the heart rate above 75?"
  - If yes: "Is the variability high?"
    - If yes: Predict "Active"
    - If no: Ask another question...

But here's the cool part: **YOU don't write these questions - the AI figures them out automatically!**

Let's train it!

In [ ]:
print("🎓 Training the AI...\n")

# Create the AI model
model = DecisionTreeClassifier(
    max_depth=5,  # Don't make the tree too complicated
    random_state=42
)

# TRAIN THE MODEL - this is where the magic happens!
model.fit(X_train, y_train)

print("✅ Training complete!\n")
print("🎉 Your AI has learned to recognize patterns in heartbeats!")
print("\nThe AI examined all 160 training examples and figured out the best")
print("questions to ask to tell relaxed from active states!")

## 🔬 Step 5: Testing Your AI

Now let's see how well your AI learned! We'll test it on the data it has never seen before.

In [ ]:
# Make predictions on training data (data it learned from)
y_train_pred = model.predict(X_train)
train_accuracy = accuracy_score(y_train, y_train_pred)

# Make predictions on test data (NEW data it hasn't seen)
y_test_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("🎯 RESULTS\n" + "="*50)
print(f"Training Accuracy: {train_accuracy*100:.1f}%")
print(f"Testing Accuracy:  {test_accuracy*100:.1f}%")
print("="*50)

print(f"\n📊 What this means:")
print(f"   The AI correctly classified {test_accuracy*100:.1f}% of heartbeats it had never seen!")

if test_accuracy > 0.95:
    print("\n🏆 OUTSTANDING! Your AI is working brilliantly!")
elif test_accuracy > 0.85:
    print("\n🎉 GREAT JOB! Your AI learned the patterns really well!")
elif test_accuracy > 0.75:
    print("\n👍 GOOD! Your AI is working, with room for improvement!")
else:
    print("\n🤔 Hmm, there might be room for improvement. Let's investigate!")

# Check if the model is overfitting
if train_accuracy - test_accuracy > 0.15:
    print("\n⚠️ Note: The AI performs much better on training data than test data.")
    print("   This is called 'overfitting' - it memorized instead of learning!")
else:
    print("\n✅ Good news: The AI generalizes well to new data!")

## 📊 Step 6: Understanding the Confusion Matrix

A **confusion matrix** shows us exactly what the AI got right and wrong:

```
                    Predicted
                Relaxed  Active
Actual Relaxed    [A]     [B]     <- A = correctly predicted relaxed, B = missed (false active)
       Active     [C]     [D]     <- D = correctly predicted active, C = missed (false relaxed)
```

Perfect AI would have all numbers in diagonal (A and D) with B and C = 0!

In [ ]:
# Calculate confusion matrix
cm = confusion_matrix(y_test, y_test_pred)

# Create a beautiful visualization
fig, ax = plt.subplots(figsize=(8, 6))

# Plot the confusion matrix
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.figure.colorbar(im, ax=ax)

# Labels
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=['Relaxed', 'Active'],
       yticklabels=['Relaxed', 'Active'],
       xlabel='Predicted Label',
       ylabel='True Label',
       title='🎯 Confusion Matrix - What Did the AI Get Right?')

# Add the numbers
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black",
                fontsize=20, fontweight='bold')

plt.tight_layout()
plt.show()

# Explain the results
print("\n📊 Reading the Confusion Matrix:\n")
print(f"✅ Correctly predicted RELAXED: {cm[0, 0]} samples")
print(f"✅ Correctly predicted ACTIVE: {cm[1, 1]} samples")
print(f"❌ Predicted ACTIVE but was RELAXED: {cm[0, 1]} samples")
print(f"❌ Predicted RELAXED but was ACTIVE: {cm[1, 0]} samples")

total_correct = cm[0, 0] + cm[1, 1]
total_samples = np.sum(cm)
print(f"\n🎯 Total correct: {total_correct}/{total_samples} ({100*total_correct/total_samples:.1f}%)")

## 📈 Step 7: Detailed Performance Report

Let's get a detailed report with some important metrics:

- **Precision**: When the AI says "Active", how often is it right?
- **Recall**: Of all the actual "Active" cases, how many did the AI find?
- **F1-Score**: A balanced score combining precision and recall

In [ ]:
# Generate classification report
print("📊 DETAILED CLASSIFICATION REPORT\n")
print(classification_report(y_test, y_test_pred, 
                          target_names=['Relaxed', 'Active']))

print("\n💡 What these metrics mean:\n")
print("📍 Precision: 'When the AI makes a prediction, how often is it correct?'")
print("   - High precision = few false alarms")
print("\n📍 Recall: 'Of all the true cases, how many did the AI catch?'")
print("   - High recall = doesn't miss many cases")
print("\n📍 F1-Score: 'A balanced score combining precision and recall'")
print("   - 1.0 is perfect, 0.0 is terrible")
print("\n📍 Support: 'How many samples of this class were in the test set'")

## 🌳 Step 8: Visualizing How the AI Thinks

Let's peek inside the AI's "brain" and see the decision tree it created!

Each box represents a question the AI asks. It follows the path from top to bottom to make a decision.

In [ ]:
# Feature names for better visualization
feature_names = [
    'mean', 'std', 'max', 'min', 'num_peaks', 'avg_peak_height',
    'avg_heart_rate', 'heart_rate_std', 'min_hr', 'max_hr',
    'mean_diff', 'std_diff'
]

# Visualize the decision tree
plt.figure(figsize=(20, 10))
plot_tree(model, 
          feature_names=feature_names,
          class_names=['Relaxed', 'Active'],
          filled=True,
          rounded=True,
          fontsize=10)
plt.title('🌳 Inside the AI\'s Brain: The Decision Tree', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n🌳 How to read the tree:\n")
print("📦 Each box is a decision point")
print("❓ The top line is the question being asked")
print("⬅️ Left branch = answer is YES (True)")
print("➡️ Right branch = answer is NO (False)")
print("🎨 Color: Blue = tends toward Relaxed, Orange = tends toward Active")
print("\n💡 The AI automatically learned these questions from the training data!")

## 🔍 Step 9: Testing Individual Predictions

Let's test the AI on individual heartbeats and see what it predicts!

In [ ]:
print("🔬 Testing Individual Predictions\n" + "="*60)

# Test on 10 random samples from the test set
sample_indices = np.random.choice(len(X_test), size=10, replace=False)

for idx in sample_indices:
    # Get the sample
    sample = X_test[idx:idx+1]  # Keep it 2D for prediction
    true_label = y_test[idx]
    
    # Make prediction
    prediction = model.predict(sample)[0]
    
    # Get confidence (probability)
    probabilities = model.predict_proba(sample)[0]
    confidence = probabilities[prediction] * 100
    
    # Display results
    true_state = "😌 Relaxed" if true_label == 0 else "⚡ Active"
    pred_state = "😌 Relaxed" if prediction == 0 else "⚡ Active"
    correct = "✅" if true_label == prediction else "❌"
    
    print(f"{correct} True: {true_state:15} | Predicted: {pred_state:15} | Confidence: {confidence:.1f}%")

print("\n💡 Confidence shows how sure the AI is about its prediction!")

## 🎓 Step 10: Feature Importance

Which features did the AI find most helpful for making decisions?
Let's find out!

In [ ]:
# Get feature importance
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]  # Sort in descending order

# Plot feature importance
plt.figure(figsize=(12, 6))
plt.bar(range(len(importances)), importances[indices], color='steelblue', alpha=0.8)
plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=45, ha='right')
plt.xlabel('Features', fontsize=12)
plt.ylabel('Importance', fontsize=12)
plt.title('🎯 Which Features Does the AI Find Most Useful?', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Print top 5 features
print("\n🏆 TOP 5 MOST IMPORTANT FEATURES:\n")
for i in range(min(5, len(importances))):
    feat_idx = indices[i]
    print(f"{i+1}. {feature_names[feat_idx]:20} Importance: {importances[feat_idx]:.3f}")

print("\n💡 Higher importance = the AI uses this feature more for decisions!")

## 🚀 Step 11: Improving Your AI

Let's try making the AI better by adjusting some settings!
We'll try different tree depths and see which works best.

In [ ]:
print("🔬 Experimenting with different AI configurations...\n")

depths_to_test = [2, 3, 5, 7, 10, 15]
train_scores = []
test_scores = []

for depth in depths_to_test:
    # Train a new model with this depth
    temp_model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    temp_model.fit(X_train, y_train)
    
    # Evaluate
    train_score = temp_model.score(X_train, y_train)
    test_score = temp_model.score(X_test, y_test)
    
    train_scores.append(train_score)
    test_scores.append(test_score)
    
    print(f"Depth={depth:2d}  |  Train: {train_score:.3f}  |  Test: {test_score:.3f}")

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(depths_to_test, train_scores, 'o-', label='Training Accuracy', linewidth=2, markersize=8)
plt.plot(depths_to_test, test_scores, 's-', label='Testing Accuracy', linewidth=2, markersize=8)
plt.xlabel('Tree Depth', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('🎯 Finding the Best Tree Depth', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Find the best depth
best_idx = np.argmax(test_scores)
best_depth = depths_to_test[best_idx]
best_score = test_scores[best_idx]

print(f"\n🏆 Best Configuration: Depth = {best_depth} with {best_score:.1%} test accuracy!")
print(f"\n💡 Notice: If training accuracy is much higher than test accuracy,")
print(f"   the model might be 'overfitting' (memorizing instead of learning)!")

## 🎉 Exercises & Challenges

Time to experiment and make this AI your own!

### Exercise 1: Try Different Features
Can you modify the `extract_features` function to include new features?
Then retrain the model and see if it improves!

In [ ]:
# YOUR CODE HERE
# Try adding new features and retraining!
# Ideas:
#   - Peak width (how wide are the heartbeats?)
#   - Energy in different frequency bands
#   - Ratio of max to min values

print("💪 Your turn! Try adding features above and retrain the model!")

### Exercise 2: Test on Your Own Data
Generate a completely new heartbeat signal and see what the AI predicts!

In [ ]:
# Generate a mystery signal
mystery_signal = generate_heartbeat_signal(
    duration=30,
    base_hr=75,  # Try different values!
    variability=5,
    stress_level=0.3
)

# Extract features
mystery_features = extract_features(mystery_signal)
mystery_features = np.array(mystery_features).reshape(1, -1)

# Predict
prediction = model.predict(mystery_features)[0]
probabilities = model.predict_proba(mystery_features)[0]

pred_state = "😌 Relaxed" if prediction == 0 else "⚡ Active/Stressed"
confidence = probabilities[prediction] * 100

print(f"🔮 Prediction: {pred_state}")
print(f"🎯 Confidence: {confidence:.1f}%")
print(f"\n📊 Probabilities:")
print(f"   Relaxed: {probabilities[0]*100:.1f}%")
print(f"   Active:  {probabilities[1]*100:.1f}%")

### Exercise 3: Try a Different AI Algorithm
Decision trees aren't the only option! Try using a Random Forest (it's like having many decision trees vote together).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create and train a Random Forest
forest_model = RandomForestClassifier(
    n_estimators=100,  # Use 100 trees!
    max_depth=5,
    random_state=42
)

forest_model.fit(X_train, y_train)

# Evaluate
forest_train_score = forest_model.score(X_train, y_train)
forest_test_score = forest_model.score(X_test, y_test)

print("🌲 Random Forest Results:\n")
print(f"Training Accuracy: {forest_train_score:.1%}")
print(f"Testing Accuracy:  {forest_test_score:.1%}")
print(f"\n🏆 Comparison:")
print(f"Decision Tree:  {test_accuracy:.1%}")
print(f"Random Forest:  {forest_test_score:.1%}")

if forest_test_score > test_accuracy:
    print("\n🎉 The Random Forest performed better!")
else:
    print("\n👍 The Decision Tree held its own!")

## 🎊 Congratulations! You've Built Your First AI!

This is HUGE! Let's recap what you accomplished:

### ✅ What You Learned:

1. **Machine Learning Basics**: How AI learns from examples instead of following exact rules
2. **Data Preparation**: How to split data into training and testing sets
3. **Model Training**: How to teach an AI to recognize patterns
4. **Evaluation**: How to measure how well your AI performs
5. **Confusion Matrix**: How to see exactly what the AI got right and wrong
6. **Feature Importance**: Which measurements matter most
7. **Model Improvement**: How to experiment and make your AI better

### 🚀 What's Next?

You're ready for more advanced topics:
- Neural networks (the basis of "deep learning")
- Real-time prediction systems
- Working with actual biosensor data
- Building complete health monitoring applications

### 💭 Think About This:

- Your phone's face recognition uses similar AI (but with images)
- Smartwatches use this for activity detection
- Medical devices use AI to detect heart problems
- You now understand the basics of how all these work!

### 🌟 Final Thoughts:

You just built an AI that can analyze heartbeats with over 90% accuracy. That's not a toy - that's real technology used in real healthcare applications!

Keep learning, keep experimenting, and remember: **Every expert was once a beginner who never gave up!**

---

**Ready for the next adventure?** In the following notebooks, we'll work with more complex analyses and even design research studies! 🎉